<a href="https://colab.research.google.com/github/deboleenadebroy/StockRight/blob/main/00_setup_and_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# [NB-00 | Cell 1] Connect Google Drive to Colab

from google.colab import drive

drive.mount('/content/drive')

print("Google Drive connected successfully!")


Mounted at /content/drive
Google Drive connected successfully!


In [ ]:

# [NB-00 | Cell 2] Create the StockRight project folders

import os

BASE = '/content/drive/MyDrive/StockRight'

folders = [
    'src',
    'raw/dailyCoal1',
    'raw/dgr2',
    'raw/monthCoal1',
    'raw/idp',
    'interim',
    'processed',
    'outputs/figures',
    'outputs/tables',
    'models',
    'logs',
    'app/data',
    'notebooks'
]

for folder in folders:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

print("StockRight project folders created.")
print("Project location:", BASE)
print("Number of folders:", len(folders))

print("\nTop-level contents:")
print(sorted(os.listdir(BASE)))


StockRight project folders created.
Project location: /content/drive/MyDrive/StockRight
Number of folders: 13

Top-level contents:
['app', 'interim', 'logs', 'models', 'notebooks', 'outputs', 'processed', 'raw', 'src']


In [ ]:

# [NB-00 | Cell 3] Test access to the CEA daily report

import requests

url = (
    "https://npp.gov.in/public-reports/cea/daily/fuel/"
    "04-10-2026/dailyCoal1-2026-10-04.xls"
)

try:
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    print("HTTP status:", response.status_code)
    print("Downloaded bytes:", len(response.content))
    print("First four bytes:", response.content[:4])

    if response.status_code == 200 and response.content[:4] == b'\xd0\xcf\x11\xe0':
        print("RESULT: CEA Excel file is accessible from Colab.")
    else:
        print("RESULT: File not verified. We need to investigate access.")

except requests.RequestException as error:
    print("RESULT: Request failed.")
    print("Error:", error)


HTTP status: 200
Downloaded bytes: 159232
First four bytes: b'\xd0\xcf\x11\xe0'
RESULT: CEA Excel file is accessible from Colab.


In [1]:
# [NB-00 | Cell 4] Download a small sample of CEA reports

import os
import requests
from pathlib import Path

BASE = "/content/drive/MyDrive/StockRight"

# Automatically create required folders
for folder in [
    "raw/dailyCoal1",
    "raw/dgr2",
    "raw/monthCoal1",
    "logs"
]:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

# CEA daily coal-stock report for the tested date
url = (
    "https://npp.gov.in/public-reports/cea/daily/fuel/"
    "04-10-2026/dailyCoal1-2026-10-04.xls"
)

output_path = os.path.join(
    BASE, "raw", "dailyCoal1", "dailyCoal1-2026-10-04.xls"
)

response = requests.get(url, timeout=60)
response.raise_for_status()

# Confirm that the downloaded content is an Excel XLS file
if not response.content.startswith(b"\xd0\xcf\x11\xe0"):
    raise ValueError(
        "The response is not a recognised XLS file. "
        "Do not save it as an Excel report."
    )

with open(output_path, "wb") as file:
    file.write(response.content)

print("Sample download completed.")
print("Saved to:", output_path)
print("File size:", f"{len(response.content):,}", "bytes")
print("File exists:", os.path.isfile(output_path))




Sample download completed.
Saved to: /content/drive/MyDrive/StockRight/raw/dailyCoal1/dailyCoal1-2026-10-04.xls
File size: 159,232 bytes
File exists: True

Folder exists: True
Excel reports currently present: 1


In [ ]:

# [NB-00 | Cell 5] Inspect the downloaded CEA Excel report

%pip -q install xlrd==2.0.1

import os
import pandas as pd

BASE = "/content/drive/MyDrive/StockRight"
FILE_PATH = os.path.join(
    BASE, "raw", "dailyCoal1", "dailyCoal1-2026-10-04.xls"
)

if not os.path.isfile(FILE_PATH):
    raise FileNotFoundError(f"Report not found: {FILE_PATH}")

# Identify all worksheets in the report
excel_file = pd.ExcelFile(FILE_PATH, engine="xlrd")

print("REPORT:", os.path.basename(FILE_PATH))
print("WORKSHEETS:", excel_file.sheet_names)

# Display a small preview of each worksheet
for sheet in excel_file.sheet_names:
    print("\n" + "=" * 70)
    print("SHEET:", sheet)

    preview = pd.read_excel(
        FILE_PATH,
        sheet_name=sheet,
        header=None,
        nrows=15,
        engine="xlrd"
    )

    print("Preview dimensions:", preview.shape)
    display(preview)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 3.9 MB/s eta 0:00:00
REPORT: dailyCoal1-2026-10-04.xls
WORKSHEETS: ['dailyCoalReport']

SHEET: dailyCoalReport
Preview dimensions: (15, 40)


,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,39
0,CENTRAL ELECTRICITY AUTHORITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FUEL MANAGEMENT DIVISION,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(AS ON 04-10-2026),NaN
3,Sl. No./State,State,GENCO,NaN,Mode of Transport,NaN,Name of Thermal Power Station/Performance of U...,NaN,Capacity (MW),Current Month PLF (Tentative),...,NaN,Receipt of the day('000 T),Consumption of the day ('000 T),NaN,NaN,Reasons for critical coal stock/Remarks,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,2,NaN,3.0,4,NaN,5,NaN,6,7,...,NaN,16,17,NaN,NaN,18,NaN,NaN,NaN,NaN
6,"A. PLANT HAVING COAL LINKAGES, NO LINKAGES AND...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,Haryana,HPGCL,NaN,RAIL,NaN,PANIPAT TPS,NaN,710,82%,...,NaN,11.8,8.1,NaN,NaN,BCCL/CCL to ensure coal supply as per subgroup...,NaN,NaN,NaN,NaN
9,2,Haryana,HPGCL,NaN,RAIL,NaN,RAJIV GANDHI TPS,NaN,1200,66%,...,NaN,19.3,16.1,NaN,NaN,BCCL/CCL to ensure coal supply as per subgroup...,NaN,NaN,NaN,NaN


In [ ]:

# [NB-00 | Cell 6] Map CEA report columns and inspect row structure

import os
import pandas as pd

FILE_PATH = (
    "/content/drive/MyDrive/StockRight/raw/dailyCoal1/"
    "dailyCoal1-2026-10-04.xls"
)

df_raw = pd.read_excel(
    FILE_PATH,
    sheet_name="dailyCoalReport",
    header=None,
    engine="xlrd"
)

print("Full report dimensions:", df_raw.shape)

# Print every column's values from the header rows
print("\nCOLUMN MAP — HEADER ROWS")
for col in range(df_raw.shape[1]):
    values = []
    for row in range(min(6, len(df_raw))):
        value = df_raw.iat[row, col]
        if pd.notna(value):
            values.append(f"row {row}: {str(value).strip()}")
    print(f"Column {col:02d}: " + " | ".join(values))

# Print non-empty cells for the first 30 rows
print("\nROW STRUCTURE — FIRST 30 ROWS")
for row in range(min(30, len(df_raw))):
    cells = []
    for col in range(df_raw.shape[1]):
        value = df_raw.iat[row, col]
        if pd.notna(value) and str(value).strip():
            cells.append(f"C{col}={str(value).strip()}")
    print(f"Row {row:03d}: " + " | ".join(cells))


Full report dimensions: (319, 40)

COLUMN MAP — HEADER ROWS
Column 00: row 0: CENTRAL ELECTRICITY AUTHORITY | row 1: FUEL MANAGEMENT DIVISION | row 3: Sl. No./State | row 5: 1
Column 01: row 3: State | row 5: 2
Column 02: row 3: GENCO
Column 03: row 5: 3.0
Column 04: row 3: Mode of Transport | row 5: 4
Column 05: 
Column 06: row 3: Name of Thermal Power Station/Performance of Utility # | row 5: 5
Column 07: 
Column 08: row 3: Capacity (MW) | row 5: 6
Column 09: row 3: Current Month PLF (Tentative) | row 5: 7
Column 10: 
Column 11: 
Column 12: row 3: Normative Stock Reqd.(Days) | row 5: 8
Column 13: 
Column 14: row 3: Daily Requirement @85% PLF (In '000 Tonnes) | row 5: 9
Column 15: 
Column 16: 
Column 17: row 3: Normative Stock Reqd.(In '000 Tonnes') | row 5: 10
Column 18: row 3: Actual Stock (In ‘000 Tonnes) | row 4: Indigenous | row 5: 11
Column 19: 
Column 20: 
Column 21: row 4: Import
Column 22: 
Column 23: row 5: 13.0
Column 24: row 4: Total
Column 25: 
Column 26: 
Column 27: row 

In [ ]:

# [NB-00 | Cell 7] Check plant rows, totals, and critical indicators

import pandas as pd
import os

FILE_PATH = (
    "/content/drive/MyDrive/StockRight/raw/dailyCoal1/"
    "dailyCoal1-2026-10-04.xls"
)

raw = pd.read_excel(
    FILE_PATH,
    sheet_name="dailyCoalReport",
    header=None,
    engine="xlrd"
)

# Extract relevant columns without changing the original worksheet
check = pd.DataFrame({
    "serial": raw.iloc[:, 0],
    "state": raw.iloc[:, 1],
    "genco": raw.iloc[:, 2],
    "plant": raw.iloc[:, 6],
    "total_stock_kt": raw.iloc[:, 24],
    "critical_marker": raw.iloc[:, 29],
    "receipts_kt": raw.iloc[:, 31],
    "consumption_kt": raw.iloc[:, 32],
})

# Normalize text fields for classification only
for col in ["state", "genco", "plant", "critical_marker"]:
    check[col] = check[col].astype("string").str.strip()

check["plant"] = check["plant"].replace("", pd.NA)

# Identify company-summary rows and section headings
is_total = check["plant"].str.contains(
    r"-Total$", case=False, na=False
)

is_section = (
    check["plant"].str.contains(
        r"^(A\.|B\.|C\.|D\.)", case=False, na=False
    )
    | check["state"].isna()
    & check["genco"].isna()
    & check["plant"].isna()
)

# Potential plant records have a plant name and a numeric serial number
serial_num = pd.to_numeric(check["serial"], errors="coerce")
is_numbered = serial_num.notna()

is_plant = (
    is_numbered
    & check["plant"].notna()
    & ~is_total
    & ~is_section
)

print("TOTAL WORKSHEET ROWS:", len(check))
print("Potential numbered plant rows:", int(is_plant.sum()))
print("Company-summary rows:", int(is_total.sum()))
print("Blank or section-like rows:", int(is_section.sum()))

print("\nCRITICAL MARKER COUNTS")
print(check["critical_marker"].value_counts(dropna=False).to_string())

print("\nSAMPLE COMPANY-SUMMARY ROWS")
display(check.loc[is_total].head(10))

print("\nSAMPLE POTENTIAL PLANT ROWS")
display(check.loc[
    is_plant,
    [
        "serial", "state", "genco", "plant",
        "total_stock_kt", "critical_marker",
        "receipts_kt", "consumption_kt"
    ]
].head(10))

print("\nPOTENTIAL PLANT ROWS WITH MISSING KEY VALUES")
key_cols = ["state", "genco", "plant", "total_stock_kt"]
display(
    check.loc[is_plant & check[key_cols].isna().any(axis=1), key_cols]
    .head(20)
)


TOTAL WORKSHEET ROWS: 319
Potential numbered plant rows: 191
Company-summary rows: 33
Blank or section-like rows: 80

CRITICAL MARKER COUNTS
critical_marker
<NA>            184
*                92
1                15
0                 8
15                2
7                 2
3                 2
8                 2
6                 2
Critical (*)      1
4                 1
2                 1
33                1
19                1
9                 1
31                1
83                1
84                1
92                1

SAMPLE COMPANY-SUMMARY ROWS


/tmp/ipykernel_2043/3490059524.py:42: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  check["plant"].str.contains(


,serial,state,genco,plant,total_stock_kt,critical_marker,receipts_kt,consumption_kt
11,NaN,<NA>,<NA>,HPGCL-Total,113.2,3,35.3,33.1
17,NaN,<NA>,<NA>,PSPCL-Total,712,0,8.2,23.8
27,NaN,<NA>,<NA>,RRVUNL-Total,227.6,7,78.6,92.3
36,NaN,<NA>,<NA>,UPRVUNL-Total,642.8,4,86.2,96.6
42,NaN,<NA>,<NA>,CSPGCL-Total,516.9,0,30,45
48,NaN,<NA>,<NA>,GSECL-Total,490,1,28.1,39.5
55,NaN,<NA>,<NA>,MPPGCL-Total,412.7,0,60.1,54.1
65,NaN,<NA>,<NA>,MAHAGENCO-Total,689,6,141.1,136
70,NaN,<NA>,<NA>,APGENCO-Total,209.6,2,42.5,55.5
74,NaN,<NA>,<NA>,APPDCL-Total,81.8,1,22.6,26.5



SAMPLE POTENTIAL PLANT ROWS


,serial,state,genco,plant,total_stock_kt,critical_marker,receipts_kt,consumption_kt
5,1,2,<NA>,5,NaN,15,16,17
8,1,Haryana,HPGCL,PANIPAT TPS,31.3,*,11.8,8.1
9,2,Haryana,HPGCL,RAJIV GANDHI TPS,36.2,*,19.3,16.1
10,3,Haryana,HPGCL,YAMUNA NAGAR TPS,45.7,*,4.2,8.9
14,4,Punjab,PSPCL,GH TPS (LEH.MOH.),133.1,<NA>,4,12
15,5,Punjab,PSPCL,GOINDWAL SAHIB TPP,158.6,<NA>,4.2,5.9
16,6,Punjab,PSPCL,ROPAR TPS,420.3,<NA>,0,5.9
20,7,Rajasthan,RRVUNL,CHHABRA-II TPP,28.1,*,11.9,15.4
21,8,Rajasthan,RRVUNL,CHHABRA-I PH-1 TPP,24.5,*,4.1,7.1
22,9,Rajasthan,RRVUNL,CHHABRA-I PH-2 TPP,14.9,*,4.1,7.4



POTENTIAL PLANT ROWS WITH MISSING KEY VALUES


,state,genco,plant,total_stock_kt
5,2,<NA>,5,NaN


In [2]:

# [NB-00 | Cell 8] Recover historical CEA daily coal reports

import csv
import time
import datetime as dt
from pathlib import Path

import requests

# --------------------------------------------------
# 1. Paths and folders
# --------------------------------------------------

BASE = Path("/content/drive/MyDrive/StockRight")
OUT = BASE / "raw" / "dailyCoal1"

# Separate recovery log: preserve the original log
LOG = BASE / "logs" / "download_log_recovery.csv"

OUT.mkdir(parents=True, exist_ok=True)
LOG.parent.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Date range
# --------------------------------------------------

START = dt.date(2024, 1, 1)
END = dt.date(2026, 10, 18)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (StockRight data collection)"
}

MAGIC = {
    "xls": b"\xd0\xcf\x11\xe0",
    "pdf": b"%PDF"
}

# --------------------------------------------------
# 3. Session and recovery log
# --------------------------------------------------

session = requests.Session()
session.headers.update(HEADERS)

log_columns = ["report", "date", "status", "file_path", "details"]

# Create the header only if the recovery log is new or empty
if not LOG.exists() or LOG.stat().st_size == 0:
    with open(LOG, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(log_columns)

counts = {
    "downloaded_xls": 0,
    "downloaded_pdf": 0,
    "already_present": 0,
    "missing": 0,
    "network_errors": 0
}

# --------------------------------------------------
# 4. Download one report without overwriting files
# --------------------------------------------------

def download_daily_report(day):
    filename_base = f"dailyCoal1-{day:%Y-%m-%d}"
    errors = []

    # Preserve reports already downloaded
    for ext in ("xls", "pdf"):
        existing = OUT / f"{filename_base}.{ext}"
        if existing.exists():
            return "already_present", str(existing), ""

    had_network_error = False

    for ext in ("xls", "pdf"):
        url = (
            "https://npp.gov.in/public-reports/cea/daily/fuel/"
            f"{day:%d-%m-%Y}/{filename_base}.{ext}"
        )

        try:
            response = session.get(url, timeout=45)

            if (
                response.status_code == 200
                and response.content.startswith(MAGIC[ext])
            ):
                destination = OUT / f"{filename_base}.{ext}"

                # Do not overwrite a file if it appeared meanwhile
                if destination.exists():
                    return "already_present", str(destination), ""

                destination.write_bytes(response.content)

                return f"downloaded_{ext}", str(destination), ""

            errors.append(
                f"{ext}: HTTP {response.status_code}, "
                "or content was not a valid expected file"
            )

        except requests.RequestException as e:
            had_network_error = True
            errors.append(f"{ext}: {type(e).__name__}: {e}")

        time.sleep(1)

    details = " | ".join(errors)

    if had_network_error:
        return "network_errors", "", details

    return "missing", "", details

# --------------------------------------------------
# 5. Download all dates and record each result
# --------------------------------------------------

total_days = (END - START).days + 1

print(f"Collection period: {START} to {END}")
print(f"Dates to check: {total_days}")
print("Destination:", OUT)
print("Recovery log:", LOG)
print("Existing reports will not be overwritten.")
print("Keep Colab running until the cell finishes.\n")

with open(LOG, "a", newline="", encoding="utf-8") as log_file:
    writer = csv.writer(log_file)

    day = START
    checked = 0

    while day <= END:
        status, saved_path, details = download_daily_report(day)

        writer.writerow([
            "dailyCoal1",
            day.isoformat(),
            status,
            saved_path,
            details
        ])
        log_file.flush()

        counts[status] += 1
        checked += 1

        if checked % 30 == 0 or day == END:
            print(
                f"Progress: {checked}/{total_days} | "
                f"XLS: {counts['downloaded_xls']} | "
                f"PDF: {counts['downloaded_pdf']} | "
                f"Already present: {counts['already_present']} | "
                f"Missing: {counts['missing']} | "
                f"Network errors: {counts['network_errors']}"
            )

        # Be considerate to the source website
        time.sleep(2.5)
        day += dt.timedelta(days=1)

# --------------------------------------------------
# 6. Final verification
# --------------------------------------------------

saved_xls = list(OUT.glob("dailyCoal1-*.xls"))
saved_pdf = list(OUT.glob("dailyCoal1-*.pdf"))

print("\nDAILY CEA RECOVERY FINISHED")
print("-" * 45)
print("Download attempt summary:", counts)
print("XLS files currently in folder:", len(saved_xls))
print("PDF files currently in folder:", len(saved_pdf))
print("Raw report folder:", OUT)
print("Recovery log:", LOG)


Collection period: 2024-01-01 to 2026-10-18
Dates to check: 1022
Destination: /content/drive/MyDrive/StockRight/raw/dailyCoal1
Recovery log: /content/drive/MyDrive/StockRight/logs/download_log_recovery.csv
Existing reports will not be overwritten.
Keep Colab running until the cell finishes.

Progress: 30/1022 | XLS: 0 | PDF: 0 | Already present: 0 | Missing: 30 | Network errors: 0
Progress: 60/1022 | XLS: 0 | PDF: 0 | Already present: 0 | Missing: 60 | Network errors: 0
Progress: 90/1022 | XLS: 0 | PDF: 0 | Already present: 0 | Missing: 90 | Network errors: 0
Progress: 120/1022 | XLS: 0 | PDF: 0 | Already present: 0 | Missing: 120 | Network errors: 0
Progress: 150/1022 | XLS: 0 | PDF: 0 | Already present: 0 | Missing: 150 | Network errors: 0
Progress: 180/1022 | XLS: 23 | PDF: 0 | Already present: 0 | Missing: 157 | Network errors: 0
Progress: 210/1022 | XLS: 52 | PDF: 0 | Already present: 0 | Missing: 158 | Network errors: 0
Progress: 240/1022 | XLS: 82 | PDF: 0 | Already present: 0 |

In [4]:

# [NB-00 | Cell 9] Verify restored CEA daily reports

from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/drive/MyDrive/StockRight")
REPORT_DIR = BASE / "raw" / "dailyCoal1"

# Prefer the recovery log; fall back to the original log
RECOVERY_LOG = BASE / "logs" / "download_log_recovery.csv"
ORIGINAL_LOG = BASE / "logs" / "download_log.csv"

LOG_FILE = (
    RECOVERY_LOG if RECOVERY_LOG.exists()
    else ORIGINAL_LOG
)

# Find downloaded reports
files = [
    f for f in REPORT_DIR.glob("dailyCoal1-*")
    if f.suffix.lower() in {".xls", ".pdf"}
]

# Extract report dates from filenames
report_dates = []

for file in files:
    match = re.search(
        r"dailyCoal1-(\d{4}-\d{2}-\d{2})",
        file.name
    )
    if match:
        report_dates.append(
            pd.to_datetime(match.group(1), errors="coerce")
        )

valid_dates = pd.Series(report_dates).dropna()

print("FILES ACTUALLY SAVED")
print("-" * 40)
print("Total files:", len(files))
print("XLS files:", sum(f.suffix.lower() == ".xls" for f in files))
print("PDF files:", sum(f.suffix.lower() == ".pdf" for f in files))

if not valid_dates.empty:
    print("Earliest date:", valid_dates.min().date())
    print("Latest date:", valid_dates.max().date())
    print("Unique report dates:", valid_dates.nunique())

# Check the selected download log
print("\nDOWNLOAD LOG")
print("-" * 40)

if LOG_FILE.exists():
    print("Log file:", LOG_FILE.name)

    log = pd.read_csv(LOG_FILE)
    print("Log entries:", len(log))

    if "status" in log.columns:
        print("\nStatus counts:")
        print(log["status"].value_counts(dropna=False).to_string())
    else:
        print("The log does not contain a 'status' column.")
else:
    print("No download log found.")

print("\nVerification complete. No files were modified.")


FILES ACTUALLY SAVED
----------------------------------------
Total files: 835
XLS files: 835
PDF files: 0
Earliest date: 2024-06-05
Latest date: 2026-10-17
Unique report dates: 835

DOWNLOAD LOG
----------------------------------------
Log file: download_log_recovery.csv
Log entries: 1022

Status counts:
status
downloaded_xls     834
missing            187
already_present      1

Verification complete. No files were modified.


In [ ]:

# [NB-00 | Cell 10] Check India Data Portal file

from pathlib import Path

BASE = Path("/content/drive/MyDrive/StockRight")
IDP_DIR = BASE / "raw" / "idp"

# Automatically create the folder if needed
IDP_DIR.mkdir(parents=True, exist_ok=True)

csv_files = list(IDP_DIR.glob("*.csv"))

print("India Data Portal folder:", IDP_DIR)
print("CSV files found:", len(csv_files))

if csv_files:
    for file in csv_files:
        print("\nFilename:", file.name)
        print("Size (MB):", round(file.stat().st_size / (1024 * 1024), 2))
else:
    print("\nNo CSV found yet.")
    print("Download the 'Daily Coal Stocks' resource and save it here.")


India Data Portal folder: /content/drive/MyDrive/StockRight/raw/idp
CSV files found: 1

Filename: daily-coal-stocks.csv
Size (MB): 49.79


In [ ]:

# [NB-00 | Cell 11] Inspect India Data Portal CSV

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/StockRight")
CSV_PATH = BASE / "raw" / "idp" / "daily-coal-stocks.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

print("File:", CSV_PATH.name)
print("Size (MB):", round(CSV_PATH.stat().st_size / (1024 * 1024), 2))

# Read the column headers
headers = pd.read_csv(CSV_PATH, nrows=0, encoding="utf-8-sig")
print("\nNUMBER OF COLUMNS:", len(headers.columns))
print("\nCOLUMN NAMES:")
for i, column in enumerate(headers.columns, start=1):
    print(f"{i}. {column}")

# Display a small sample only
sample = pd.read_csv(CSV_PATH, nrows=5, encoding="utf-8-sig")

print("\nFIRST FIVE RECORDS:")
display(sample)

print("\nCSV inspection complete. Original file unchanged.")


File: daily-coal-stocks.csv
Size (MB): 49.79

NUMBER OF COLUMNS: 22

COLUMN NAMES:
1. id
2. date
3. state_name
4. state_code
5. power_station_name
6. sector
7. utility
8. mode_of_transport
9. capacity
10. daily_requirement
11. daily_receipt
12. daily_consumption
13. req_normative_stock
14. normative_stock_days
15. indigenous_stock
16. import_stock
17. total_stock
18. stock_days
19. plf_prcnt
20. actual_vs_normative_stock_prcnt
21. is_critical
22. remarks

FIRST FIVE RECORDS:


,id,date,state_name,state_code,power_station_name,sector,utility,mode_of_transport,capacity,daily_requirement,...,req_normative_stock,normative_stock_days,indigenous_stock,import_stock,total_stock,stock_days,plf_prcnt,actual_vs_normative_stock_prcnt,is_critical,remarks
0,0,2018-11-27,Andhra Pradesh,28,Damodaram Sanjeevaiah Tps,State Sector,APGENCO,Rail,1600,15.95,...,NaN,20,26.45,59.74,86.19,5,46.68,NaN,NaN,CIL Supply 98% of prorata ACQ.
1,1,2018-11-27,Andhra Pradesh,28,Dr. N.Tata Rao Tps,State Sector,APGENCO,Rail,1760,20.59,...,NaN,20,93.39,0.00,93.39,5,67.63,NaN,Critical,Supply to APGENCO 84% prorata ACQ
2,2,2018-11-27,Andhra Pradesh,28,Painampuram Tpp,Pvt Sector,IPP,Rail,1320,11.33,...,NaN,20,37.58,7.97,45.55,4,60.21,NaN,Critical,CIL Supply 69% of prorata ACQ.
3,3,2018-11-27,Andhra Pradesh,28,Rayalaseema Tps,State Sector,APGENCO,Rail,1650,9.01,...,NaN,25,125.22,0.00,125.22,14,37.64,NaN,NaN,NaN
4,4,2018-11-27,Andhra Pradesh,28,Simhadri,Central Sector,NTPC,Rail,2000,23.79,...,NaN,25,0.00,0.00,0.00,0,66.02,NaN,NaN,Supply to NTPC 99% of prorata ACQ



CSV inspection complete. Original file unchanged.


In [ ]:

# [NB-00 | Cell 12] Verify India Data Portal date coverage

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/StockRight")
CSV_PATH = BASE / "raw" / "idp" / "daily-coal-stocks.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

# Read only the date column to keep this check lightweight
dates_df = pd.read_csv(
    CSV_PATH,
    usecols=["date"],
    encoding="utf-8-sig"
)

dates = pd.to_datetime(dates_df["date"], errors="coerce")

print("TOTAL RECORDS:", len(dates_df))
print("Invalid or missing dates:", int(dates.isna().sum()))

valid_dates = dates.dropna()

if not valid_dates.empty:
    print("Earliest date:", valid_dates.min().date())
    print("Latest date:", valid_dates.max().date())
    print("Unique report dates:", valid_dates.dt.date.nunique())

    # Calendar gaps are reported for awareness, not assumed to be errors
    all_dates = pd.date_range(
        valid_dates.min().normalize(),
        valid_dates.max().normalize(),
        freq="D"
    )
    available_dates = pd.DatetimeIndex(
        valid_dates.dt.normalize().unique()
    )
    missing_dates = all_dates.difference(available_dates)

    print("Calendar dates in range:", len(all_dates))
    print("Dates without any records:", len(missing_dates))
    print("First 15 dates without records:")
    print([d.date().isoformat() for d in missing_dates[:15]])

print("\nOriginal CSV remains unchanged.")


TOTAL RECORDS: 362035
Invalid or missing dates: 0
Earliest date: 2018-11-27
Latest date: 2025-03-03
Unique report dates: 2277
Calendar dates in range: 2289
Dates without any records: 12
First 15 dates without records:
['2019-01-24', '2019-07-29', '2020-03-20', '2020-03-21', '2021-06-25', '2021-06-26', '2022-01-28', '2022-01-29', '2022-01-30', '2024-11-01', '2025-01-31', '2025-02-04']

Original CSV remains unchanged.


In [ ]:
# [NB-00 | Cell 13] Test one CEA daily generation report

from pathlib import Path
import requests

BASE = Path("/content/drive/MyDrive/StockRight")
RAW_DIR = BASE / "raw" / "dgr2"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Test report for 4 October 2026
url = (
    "https://npp.gov.in/public-reports/cea/daily/dgr/"
    "04-10-2026/dgr2-2026-10-04.xls"
)

try:
    response = requests.get(url, timeout=30)

    print("HTTP status:", response.status_code)
    print("Response size:", len(response.content), "bytes")
    print("Content type:", response.headers.get("Content-Type"))

    if response.status_code == 200 and response.content[:4] == bytes.fromhex("D0CF11E0"):
        output_path = RAW_DIR / "dgr2-2026-10-04.xls"
        output_path.write_bytes(response.content)

        print("RESULT: Excel report downloaded successfully.")
        print("Saved to:", output_path)
    else:
        print("RESULT: The XLS test was unsuccessful.")
        print("We will check the official report format before bulk downloading.")

except requests.RequestException as e:
    print("Network error:", str(e))
    print("No bulk download has been started.")

HTTP status: 200
Response size: 387072 bytes
Content type: application/vnd.ms-excel
RESULT: Excel report downloaded successfully.
Saved to: /content/drive/MyDrive/StockRight/raw/dgr2/dgr2-2026-10-04.xls


In [ ]:
# [NB-00 | Cell 14] Inspect CEA DGR2 workbook structure

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/StockRight")

FILE_PATH = (
    BASE / "raw" / "dgr2" / "dgr2-2026-10-04.xls"
)

if not FILE_PATH.exists():
    raise FileNotFoundError(f"File not found: {FILE_PATH}")

# Read the workbook without assuming any header row
workbook = pd.ExcelFile(FILE_PATH)

print("WORKBOOK INFORMATION")
print("File:", FILE_PATH.name)
print("Sheets:", workbook.sheet_names)

for sheet_name in workbook.sheet_names:
    df = pd.read_excel(
        FILE_PATH,
        sheet_name=sheet_name,
        header=None
    )

    print("\n" + "=" * 60)
    print("SHEET:", sheet_name)
    print("Dimensions:", df.shape)

    print("\nFIRST 12 ROWS:")
    display(df.head(12).iloc[:, :20])

    print("\nLAST 5 ROWS:")
    display(df.tail(5).iloc[:, :20])

print("\nInspection complete. Original file unchanged.")

WORKBOOK INFORMATION
File: dgr2-2026-10-04.xls
Sheets: ['OPM1_2']

SHEET: OPM1_2
Dimensions: (1523, 18)

FIRST 12 ROWS:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,NaN,NaN,NaN,NaN,NaN,NaN,CENTRAL ELECTRICITY AUTHORITY \n ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"REGION WISE, STATE WISE, SECTOR WISE, TYPE WIS...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Subreport-2
2,POWER STATION,NaN,NaN,NaN,NaN,NaN,NaN,MONITORED\nCAP.\n(MW),GENERATION (MU),NaN,NaN,NaN,COAL STOCK\nIN DAYS,CAP. UNDER\nOUTAGE\n(MW)\n\n,OUTAGE DATE\n(DD/MM/YYYY HH24:MI),NaN,EXPECTED DATE /\nSYNC. DATE,REMARKS
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TODAY'S\nPROGRAM,TODAY'S\nACTUAL\n,APRIL 1 TILL DATE (2026-27),NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PROG.,ACTUAL,2026-10-04 00:00:00,NaN,NaN,NaN,NaN,NaN
5,NORTHERN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,REGION TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,82086.72,1157.17,1179.99,234934.68,227943.53,NaN,14526.96,NaN,NaN,NaN,NaN
7,DELHI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,STATE TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,2100.4,11.49,23.94,2121.96,2543.15,NaN,345.8,NaN,NaN,NaN,NaN
9,SECTOR:,NaN,NaN,NaN,STATE SECTOR,NaN,NaN,2100.4,11.49,23.94,2121.96,2543.15,NaN,345.8,NaN,NaN,NaN,NaN



LAST 5 ROWS:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
1518,PUNATSANGCHHU-II,NaN,NaN,NaN,NaN,NaN,NaN,0,12.61,13.23,2627.44,2250.72,NaN,0,NaN,NaN,NaN,NaN
1519,TALA,NaN,NaN,NaN,NaN,NaN,NaN,0,0,17.53,0,1404.06,NaN,0,NaN,NaN,NaN,NaN
1520,NaN,Remarks:-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1522,NaN,Report Version :-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Inspection complete. Original file unchanged.


In [ ]:
# [NB-00 | Cell 15] Download historical CEA DGR2 reports

from pathlib import Path
from datetime import date, timedelta
import requests
import pandas as pd
import time

BASE = Path("/content/drive/MyDrive/StockRight")
RAW_DIR = BASE / "raw" / "dgr2"
LOG_DIR = BASE / "logs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = date(2024, 1, 1)
END_DATE = date(2026, 10, 18)

LOG_PATH = LOG_DIR / "dgr2_download_log.csv"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 StockRight research project"
})

dates = []
current = START_DATE

while current <= END_DATE:
    dates.append(current)
    current += timedelta(days=1)

log_rows = []
counts = {
    "downloaded_xls": 0,
    "downloaded_pdf": 0,
    "already_present": 0,
    "missing": 0,
    "network_errors": 0
}

print(f"Collection period: {START_DATE} to {END_DATE}")
print(f"Dates to check: {len(dates)}")

for i, report_date in enumerate(dates, start=1):
    date_text = report_date.strftime("%Y-%m-%d")
    folder_date = report_date.strftime("%d-%m-%Y")

    xls_path = RAW_DIR / f"dgr2-{date_text}.xls"
    pdf_path = RAW_DIR / f"dgr2-{date_text}.pdf"

    if xls_path.exists() or pdf_path.exists():
        counts["already_present"] += 1
        log_rows.append({
            "date": date_text,
            "status": "already_present",
            "file": xls_path.name if xls_path.exists() else pdf_path.name
        })
        continue

    base_url = (
        "https://npp.gov.in/public-reports/cea/daily/dgr/"
        f"{folder_date}/dgr2-{date_text}"
    )

    status = "missing"
    saved_file = ""
    network_error = False

    for extension in ("xls", "pdf"):
        try:
            response = session.get(
                f"{base_url}.{extension}",
                timeout=30
            )

            if response.status_code == 200 and len(response.content) > 1000:
                if extension == "xls":
                    # XLS files begin with the OLE signature
                    valid_file = response.content[:4] == bytes.fromhex("D0CF11E0")
                    output_path = xls_path
                else:
                    valid_file = response.content[:4] == b"%PDF"
                    output_path = pdf_path

                if valid_file:
                    output_path.write_bytes(response.content)
                    status = f"downloaded_{extension}"
                    saved_file = output_path.name
                    counts[status] += 1
                    break

        except requests.RequestException:
            network_error = True

    if status == "missing":
        if network_error:
            status = "network_errors"
            counts["network_errors"] += 1
        else:
            counts["missing"] += 1

    log_rows.append({
        "date": date_text,
        "status": status,
        "file": saved_file
    })

    # Save progress periodically in case Colab disconnects
    if i % 25 == 0 or i == len(dates):
        pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)

    if i % 50 == 0 or i == len(dates):
        print(
            f"Progress: {i}/{len(dates)} | "
            f"XLS: {counts['downloaded_xls']} | "
            f"PDF: {counts['downloaded_pdf']} | "
            f"Existing: {counts['already_present']} | "
            f"Missing: {counts['missing']} | "
            f"Network errors: {counts['network_errors']}"
        )

    # Small pause to avoid sending requests too rapidly
    time.sleep(0.05)

pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)

print("\nDGR2 DOWNLOAD ATTEMPT FINISHED")
print("Summary:", counts)
print("Raw report folder:", RAW_DIR)
print("Download log:", LOG_PATH)
print("\nNo data cleaning or modelling was performed.")

Collection period: 2024-01-01 to 2026-10-18
Dates to check: 1022
Progress: 50/1022 | XLS: 50 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 100/1022 | XLS: 100 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 150/1022 | XLS: 150 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 200/1022 | XLS: 200 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 250/1022 | XLS: 250 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 300/1022 | XLS: 300 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 350/1022 | XLS: 350 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 400/1022 | XLS: 400 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 450/1022 | XLS: 450 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 500/1022 | XLS: 500 | PDF: 0 | Existing: 0 | Missing: 0 | Network errors: 0
Progress: 550/1022 | XLS: 550 | PDF: 0 | Existing: 0 | Missing: 0 | Network e

In [ ]:
# [NB-00 | Cell 16] Verify saved DGR2 files

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/StockRight")
RAW_DIR = BASE / "raw" / "dgr2"
LOG_PATH = BASE / "logs" / "dgr2_download_log.csv"

xls_files = list(RAW_DIR.glob("*.xls"))
pdf_files = list(RAW_DIR.glob("*.pdf"))

print("SAVED FILES")
print("XLS files:", len(xls_files))
print("PDF files:", len(pdf_files))
print("Total files:", len(xls_files) + len(pdf_files))

all_files = xls_files + pdf_files
file_dates = []

for file in all_files:
    date_text = file.stem.replace("dgr2-", "")
    parsed = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
    if pd.notna(parsed):
        file_dates.append(parsed)

if file_dates:
    print("Earliest saved report:", min(file_dates).date())
    print("Latest saved report:", max(file_dates).date())
    print("Unique report dates:", len(set(file_dates)))

print("\nDOWNLOAD LOG")
if LOG_PATH.exists():
    log = pd.read_csv(LOG_PATH)
    print("Log entries:", len(log))
    print("\nStatus counts:")
    print(log["status"].value_counts().to_string())

    missing = log[log["status"] == "missing"]
    print("\nDates with no report found:", len(missing))
    print(missing["date"].to_list())
else:
    print("Download log not found.")

print("\nOriginal reports remain unchanged.")

SAVED FILES
XLS files: 1015
PDF files: 3
Total files: 1018
Earliest saved report: 2024-01-01
Latest saved report: 2026-10-14
Unique report dates: 1018

DOWNLOAD LOG
Log entries: 1022

Status counts:
status
downloaded_xls     1014
missing               4
downloaded_pdf        3
already_present       1

Dates with no report found: 4
['2026-10-15', '2026-10-16', '2026-10-17', '2026-10-18']

Original reports remain unchanged.


In [ ]:
# [NB-00 | Cell 17] Test one CEA monthly coal report

from pathlib import Path
import requests

BASE = Path("/content/drive/MyDrive/StockRight")
RAW_DIR = BASE / "raw" / "monthCoal1"
RAW_DIR.mkdir(parents=True, exist_ok=True)

url = (
    "https://npp.gov.in/public-reports/cea/monthly/fuel/"
    "2026/SEP/monthCoal1-2026-Sep.xls"
)

try:
    response = requests.get(url, timeout=30)

    print("HTTP status:", response.status_code)
    print("Response size:", len(response.content), "bytes")
    print("Content type:", response.headers.get("Content-Type"))

    if (
        response.status_code == 200
        and response.content[:4] == bytes.fromhex("D0CF11E0")
    ):
        output_path = RAW_DIR / "monthCoal1-2026-Sep.xls"
        output_path.write_bytes(response.content)

        print("RESULT: Monthly Excel report downloaded successfully.")
        print("Saved to:", output_path)
    else:
        print("RESULT: The XLS test was unsuccessful.")
        print("No bulk monthly download has been started.")

except requests.RequestException as e:
    print("Network error:", str(e))
    print("No bulk monthly download has been started.")

HTTP status: 404
Response size: 840 bytes
Content type: text/html;charset=utf-8
RESULT: The XLS test was unsuccessful.
No bulk monthly download has been started.


In [ ]:
# [NB-00 | Cell 18] Test a known historical monthly coal report

from pathlib import Path
import requests

BASE = Path("/content/drive/MyDrive/StockRight")
RAW_DIR = BASE / "raw" / "monthCoal1"
RAW_DIR.mkdir(parents=True, exist_ok=True)

base_url = (
    "https://npp.gov.in/public-reports/cea/monthly/fuel/"
    "2025/SEP/monthCoal1-2025-Sep"
)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 StockRight research"})

for extension in ("xls", "pdf"):
    url = f"{base_url}.{extension}"

    try:
        response = session.get(url, timeout=30)
        print(f"\nFormat: {extension.upper()}")
        print("HTTP status:", response.status_code)
        print("Response size:", len(response.content), "bytes")

        if response.status_code == 200:
            if extension == "xls":
                valid = response.content[:4] == bytes.fromhex("D0CF11E0")
            else:
                valid = response.content[:4] == b"%PDF"

            if valid:
                output_path = RAW_DIR / f"monthCoal1-2025-Sep.{extension}"
                output_path.write_bytes(response.content)
                print("RESULT: Valid report downloaded.")
                print("Saved to:", output_path)
                break
            else:
                print("Response was not a valid file of the expected format.")
        else:
            print("This format was not available at the tested URL.")

    except requests.RequestException as e:
        print("Network error:", str(e))

print("\nNo bulk monthly download has started.")


Format: XLS
HTTP status: 200
Response size: 93696 bytes
RESULT: Valid report downloaded.
Saved to: /content/drive/MyDrive/StockRight/raw/monthCoal1/monthCoal1-2025-Sep.xls

No bulk monthly download has started.


In [ ]:
# [NB-00 | Cell 19] Download historical CEA monthly coal reports

from pathlib import Path
import calendar
import time
import requests
import pandas as pd

BASE = Path("/content/drive/MyDrive/StockRight")
RAW_DIR = BASE / "raw" / "monthCoal1"
LOG_DIR = BASE / "logs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR, START_MONTH = 2018, 1
END_YEAR, END_MONTH = 2026, 9

LOG_PATH = LOG_DIR / "monthCoal1_download_log.csv"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 StockRight research project"
})

months = pd.period_range(
    start=f"{START_YEAR}-{START_MONTH:02d}",
    end=f"{END_YEAR}-{END_MONTH:02d}",
    freq="M"
)

log_rows = []
counts = {
    "downloaded_xls": 0,
    "downloaded_pdf": 0,
    "already_present": 0,
    "missing": 0,
    "network_errors": 0
}

print(f"Collection period: {months[0]} to {months[-1]}")
print(f"Months to check: {len(months)}")

for i, period in enumerate(months, start=1):
    year = period.year
    month_num = period.month

    folder_month = calendar.month_abbr[month_num].upper()
    filename_month = calendar.month_abbr[month_num].title()
    date_text = str(period)

    xls_path = RAW_DIR / f"monthCoal1-{year}-{filename_month}.xls"
    pdf_path = RAW_DIR / f"monthCoal1-{year}-{filename_month}.pdf"

    if xls_path.exists() or pdf_path.exists():
        counts["already_present"] += 1
        log_rows.append({
            "month": date_text,
            "status": "already_present",
            "file": xls_path.name if xls_path.exists() else pdf_path.name
        })
        continue

    base_url = (
        "https://npp.gov.in/public-reports/cea/monthly/fuel/"
        f"{year}/{folder_month}/monthCoal1-{year}-{filename_month}"
    )

    status = "missing"
    saved_file = ""
    network_error = False

    for extension in ("xls", "pdf"):
        try:
            response = session.get(
                f"{base_url}.{extension}",
                timeout=30
            )

            if response.status_code == 200 and len(response.content) > 1000:
                if extension == "xls":
                    valid_file = response.content[:4] == bytes.fromhex("D0CF11E0")
                    output_path = xls_path
                else:
                    valid_file = response.content[:4] == b"%PDF"
                    output_path = pdf_path

                if valid_file:
                    output_path.write_bytes(response.content)
                    status = f"downloaded_{extension}"
                    saved_file = output_path.name
                    counts[status] += 1
                    break

        except requests.RequestException:
            network_error = True

    if status == "missing":
        if network_error:
            status = "network_errors"
            counts["network_errors"] += 1
        else:
            counts["missing"] += 1

    log_rows.append({
        "month": date_text,
        "status": status,
        "file": saved_file
    })

    if i % 10 == 0 or i == len(months):
        pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)
        print(
            f"Progress: {i}/{len(months)} | "
            f"XLS: {counts['downloaded_xls']} | "
            f"PDF: {counts['downloaded_pdf']} | "
            f"Existing: {counts['already_present']} | "
            f"Missing: {counts['missing']} | "
            f"Network errors: {counts['network_errors']}"
        )

    time.sleep(0.1)

pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)

print("\nMONTHLY DOWNLOAD ATTEMPT FINISHED")
print("Summary:", counts)
print("Raw report folder:", RAW_DIR)
print("Download log:", LOG_PATH)
print("No cleaning, EDA, or modelling was performed.")

Collection period: 2018-01 to 2026-09
Months to check: 105
Progress: 10/105 | XLS: 0 | PDF: 0 | Existing: 0 | Missing: 10 | Network errors: 0
Progress: 20/105 | XLS: 0 | PDF: 0 | Existing: 0 | Missing: 20 | Network errors: 0
Progress: 30/105 | XLS: 0 | PDF: 0 | Existing: 0 | Missing: 30 | Network errors: 0
Progress: 40/105 | XLS: 0 | PDF: 0 | Existing: 0 | Missing: 40 | Network errors: 0
Progress: 50/105 | XLS: 0 | PDF: 0 | Existing: 0 | Missing: 50 | Network errors: 0
Progress: 60/105 | XLS: 0 | PDF: 0 | Existing: 0 | Missing: 60 | Network errors: 0
Progress: 70/105 | XLS: 7 | PDF: 0 | Existing: 0 | Missing: 63 | Network errors: 0
Progress: 80/105 | XLS: 17 | PDF: 0 | Existing: 0 | Missing: 63 | Network errors: 0
Progress: 90/105 | XLS: 27 | PDF: 0 | Existing: 0 | Missing: 63 | Network errors: 0
Progress: 100/105 | XLS: 36 | PDF: 0 | Existing: 1 | Missing: 63 | Network errors: 0
Progress: 105/105 | XLS: 40 | PDF: 0 | Existing: 1 | Missing: 64 | Network errors: 0

MONTHLY DOWNLOAD ATTE

In [ ]:
# [NB-00 | Cell 20] Verify saved monthly coal reports

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/StockRight")
RAW_DIR = BASE / "raw" / "monthCoal1"
LOG_PATH = BASE / "logs" / "monthCoal1_download_log.csv"

xls_files = list(RAW_DIR.glob("*.xls"))
pdf_files = list(RAW_DIR.glob("*.pdf"))

print("SAVED MONTHLY REPORTS")
print("XLS files:", len(xls_files))
print("PDF files:", len(pdf_files))
print("Total files:", len(xls_files) + len(pdf_files))

if xls_files or pdf_files:
    all_files = sorted(xls_files + pdf_files)
    print("First saved file:", all_files[0].name)
    print("Last saved file:", all_files[-1].name)

print("\nDOWNLOAD LOG")
if LOG_PATH.exists():
    log = pd.read_csv(LOG_PATH)

    print("Log entries:", len(log))
    print("\nStatus counts:")
    print(log["status"].value_counts().to_string())

    missing = log[log["status"] == "missing"]
    print("\nMonths not found:", len(missing))
    print(missing["month"].to_list())

    print("\nFirst 10 successful downloads:")
    successful = log[log["status"].isin(
        ["downloaded_xls", "downloaded_pdf", "already_present"]
    )]
    print(successful[["month", "status", "file"]].head(10).to_string(index=False))
else:
    print("Monthly download log not found.")

print("\nOriginal reports remain unchanged.")

SAVED MONTHLY REPORTS
XLS files: 41
PDF files: 0
Total files: 41
First saved file: monthCoal1-2023-Apr.xls
Last saved file: monthCoal1-2026-May.xls

DOWNLOAD LOG
Log entries: 105

Status counts:
status
missing            64
downloaded_xls     40
already_present     1

Months not found: 64
['2018-01', '2018-02', '2018-03', '2018-04', '2018-05', '2018-06', '2018-07', '2018-08', '2018-09', '2018-10', '2018-11', '2018-12', '2019-01', '2019-02', '2019-03', '2019-04', '2019-05', '2019-06', '2019-07', '2019-08', '2019-09', '2019-10', '2019-11', '2019-12', '2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09', '2020-10', '2020-11', '2020-12', '2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06', '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12', '2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-02', '2023-03', '2026-09']

Firs

In [ ]:
# [NB-00 | Cell 21] Inspect official CEA monthly report archive

import requests
from bs4 import BeautifulSoup

url = "https://npp.gov.in/monthlyCoalReports"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("HTTP status:", response.status_code)
print("Final URL:", response.url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")

    print("\nPAGE TITLE:")
    print(soup.title.get_text(strip=True) if soup.title else "Not found")

    print("\nFORMS:")
    for i, form in enumerate(soup.find_all("form"), start=1):
        print(f"\nForm {i}")
        print("Action:", form.get("action"))
        print("Method:", form.get("method"))

        for field in form.find_all(["input", "select", "button"]):
            print({
                "tag": field.name,
                "type": field.get("type"),
                "name": field.get("name"),
                "id": field.get("id"),
                "value": field.get("value"),
                "text": field.get_text(" ", strip=True)[:100]
            })

    print("\nLINKS MENTIONING COAL OR REPORTS:")
    for link in soup.find_all("a", href=True):
        text = link.get_text(" ", strip=True)
        href = link["href"]

        if any(word in (text + " " + href).lower()
               for word in ["coal", "monthly", "report", "archive"]):
            print(text[:100], "->", href)

else:
    print("Could not retrieve the official archive page.")

print("\nNo files were downloaded or modified.")

HTTP status: 200
Final URL: https://npp.gov.in/monthlyCoalReports

PAGE TITLE:
NPDMS: Reports

FORMS:

Form 1
Action: monthlyCoalReports
Method: post
{'tag': 'input', 'type': 'text', 'name': 'rmonth', 'id': 'rmonth', 'value': None, 'text': ''}
{'tag': 'button', 'type': 'submit', 'name': None, 'id': None, 'value': None, 'text': 'Submit'}
{'tag': 'input', 'type': 'hidden', 'name': '_csrf', 'id': None, 'value': 'DmmRWZRof8tda7_TxGtUXypf5e1dT351dMkCsj1NJn1J0bhJNlujOqxRGqpwX9yw9UZgZhNtyI8_fk1YF61n11t5FUR85Ip4', 'text': ''}

LINKS MENTIONING COAL OR REPORTS:
Daily Generation Reports -> /dgrReports
Monthly Generation Reports Actual -> /monthlyGenerationReportsAct
Monthly Generation Reports Tentative -> /monthlyGenerationReportsTen
Daily Fuel Reports -> /dailyCoalReports
Monthly Fuel Reports -> /monthlyCoalReports
Monthly Transmission Reports -> /monthlyTransmissionReports
Installed Capacity Reports -> /installedCapacityReports
Monthly Fuel Reports -> #

No files were downloaded or modified.


In [ ]:
# [NB-00 | Cell 22] Inspect monthly archive form requirements

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://npp.gov.in/monthlyCoalReports"

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

response = session.get(url, timeout=30)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")

    form = soup.find("form", attrs={"action": "monthlyCoalReports"})

    if form:
        print("\nFORM DETAILS")
        print("Action:", urljoin(url, form.get("action", "")))
        print("Method:", form.get("method", "get"))

        month_field = form.find(attrs={"name": "rmonth"})
        if month_field:
            print("\nMONTH FIELD ATTRIBUTES:")
            print(month_field.attrs)

        print("\nFORM HTML:")
        print(form.prettify()[:5000])
    else:
        print("Could not identify the expected monthly report form.")

    print("\nRELATED JAVASCRIPT CLUES")
    found = False

    for script in soup.find_all("script", src=True):
        src = urljoin(url, script["src"])
        try:
            js_response = session.get(src, timeout=15)
            if js_response.status_code == 200:
                js = js_response.text
                if "rmonth" in js or "monthlyCoalReports" in js:
                    found = True
                    print("\nScript:", src)
                    for keyword in ["rmonth", "monthlyCoalReports"]:
                        start = 0
                        while True:
                            pos = js.find(keyword, start)
                            if pos == -1:
                                break
                            print(js[max(0, pos-250):pos+400])
                            start = pos + len(keyword)
        except requests.RequestException:
            pass

    if not found:
        print("No matching clues found in externally linked scripts.")

print("\nInspection only: no reports were requested or downloaded.")

HTTP status: 200

FORM DETAILS
Action: https://npp.gov.in/monthlyCoalReports
Method: post

MONTH FIELD ATTRIBUTES:
{'name': 'rmonth', 'id': 'rmonth', 'type': 'text', 'class': ['gui-input', 'form-control'], 'placeholder': 'Select Month', 'readonly': '', 'aria-describedby': 'rmonth_help'}

FORM HTML:
<form action="monthlyCoalReports" id="form1" method="post" name="form1">
 <div class="row align-items-end">
  <div class="col-md-2 position-relative">
   <label class="field-label" for="rmonth">
    Select Month
   </label>
   <input aria-describedby="rmonth_help" class="gui-input form-control" id="rmonth" name="rmonth" placeholder="Select Month" readonly="" type="text"/>
   <span class="visually-hidden" id="rmonth_help">
    Opens a calendar
							month picker.
   </span>
  </div>
  <div class="col-md-auto">
   <button class="btn btn-primary" type="submit">
    Submit
   </button>
  </div>
 </div>
 <div>
  <input name="_csrf" type="hidden" value="EiO5SoJesQgLzLarsANgboxCr7PFejKpM4zgLmDcYV2

In [ ]:
# [NB-00 | Cell 23] Inspect the month-picker configuration

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://npp.gov.in/monthlyCoalReports"

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

response = session.get(BASE_URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

script_urls = [
    urljoin(BASE_URL, script["src"])
    for script in soup.find_all("script", src=True)
]

keywords = [
    "Zebra_DatePicker",
    "format",
    "months",
    "rmonth",
    "month",
    "datepicker"
]

for script_url in script_urls:
    try:
        result = session.get(script_url, timeout=20)
        if result.status_code != 200:
            continue

        content = result.text
        matches = []

        for keyword in keywords:
            start = 0
            while True:
                pos = content.lower().find(keyword.lower(), start)
                if pos == -1:
                    break

                snippet = content[max(0, pos - 180):pos + 300]
                if snippet not in matches:
                    matches.append(snippet)

                start = pos + len(keyword)

        if matches:
            print("\nSCRIPT:", script_url)
            for snippet in matches[:8]:
                print("-" * 50)
                print(snippet)

    except requests.RequestException as e:
        print("Could not inspect script:", script_url, str(e))

print("\nInspection complete. No reports requested or downloaded.")


SCRIPT: https://npp.gov.in/resources/assets/js/main.js
--------------------------------------------------

      if ($pickDateEl.length > 0) {
        $pickDateEl.each(function() {
          var element = $(this);

          element.find('span').html(moment().subtract(29, 'days').format('MMMM D, YYYY') + ' - ' + moment().format('MMMM D, YYYY'));

          element.daterangepicker({
            format: 'MM/DD/YYYY',
            startDate: moment().subtract(29, 'days'),
            endDate: moment(),
            minDate: '01/01/2012',
            maxDate: '12/31/2015',
--------------------------------------------------
     $pickDateEl.each(function() {
          var element = $(this);

          element.find('span').html(moment().subtract(29, 'days').format('MMMM D, YYYY') + ' - ' + moment().format('MMMM D, YYYY'));

          element.daterangepicker({
            format: 'MM/DD/YYYY',
            startDate: moment().subtract(29, 'days'),
            endDate: moment(),
            minD

In [ ]:
# [NB-00 | Cell 24] Test the official monthly archive form

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

URL = "https://npp.gov.in/monthlyCoalReports"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 StockRight research project"
})

# Open the official page to obtain the current session and CSRF token
page = session.get(URL, timeout=30)
page.raise_for_status()

soup = BeautifulSoup(page.text, "html.parser")
csrf = soup.find("input", attrs={"name": "_csrf"})

if csrf is None or not csrf.get("value"):
    raise RuntimeError("Could not obtain the official form's CSRF token.")

# Test one older month using a readable month-year value
test_month = "September 2022"

print("Testing month:", test_month)
print("Submitting official form...")

response = session.post(
    URL,
    data={
        "rmonth": test_month,
        "_csrf": csrf["value"]
    },
    headers={
        "Referer": URL,
        "Origin": "https://npp.gov.in"
    },
    timeout=30,
    allow_redirects=True
)

print("\nRESPONSE")
print("HTTP status:", response.status_code)
print("Final URL:", response.url)

result_soup = BeautifulSoup(response.text, "html.parser")
print("Page title:", result_soup.title.get_text(strip=True)
      if result_soup.title else "Not found")

print("\nPAGE TEXT CLUES")
page_text = result_soup.get_text(" ", strip=True)

for keyword in [
    "September 2022",
    "No data",
    "not found",
    "Coal",
    "Download",
    "Report"
]:
    position = page_text.lower().find(keyword.lower())
    if position >= 0:
        print(f"{keyword}: {page_text[max(0, position-120):position+250]}")

print("\nREPORT-LIKE LINKS")
for link in result_soup.find_all("a", href=True):
    href = link["href"]
    label = link.get_text(" ", strip=True)

    if any(ext in href.lower() for ext in [".xls", ".xlsx", ".pdf"]):
        print(label, "->", urljoin(response.url, href))

print("\nNo report files were saved or modified.")

Testing month: September 2022
Submitting official form...

RESPONSE
HTTP status: 200
Final URL: https://npp.gov.in/monthlyCoalReports
Page title: NPDMS: Reports

PAGE TEXT CLUES
Coal: rts Monthly Fuel Reports Monthly Transmission Reports Installed Capacity Reports Home Monthly Fuel Reports Monthly
				Coal Reports Form Select Month Opens a calendar
							month picker. Submit
Report: NPDMS: Reports Login to National Power Data Management System Reports Generation Daily Generation Reports Monthly Generation Reports Actual Monthly Generation Reports Tentative Daily Fuel Reports Monthly Fuel Reports Monthly Transmission Reports Installed C

REPORT-LIKE LINKS

No report files were saved or modified.


In [ ]:
# [NB-00 | Cell 25] Inspect month-picker initialization

import requests
from bs4 import BeautifulSoup

URL = "https://npp.gov.in/monthlyCoalReports"

response = requests.get(
    URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("PAGE:", response.url)
print("\nMONTH FIELD:")
field = soup.find("input", attrs={"name": "rmonth"})
print(field.attrs if field else "Field not found")

print("\nINLINE JAVASCRIPT REFERENCES TO THE MONTH FIELD:")

found = False

for index, script in enumerate(soup.find_all("script"), start=1):
    code = script.string or script.get_text()

    if not code:
        continue

    lines = code.splitlines()

    matches = [
        (line_number, line.strip())
        for line_number, line in enumerate(lines, start=1)
        if any(
            term in line.lower()
            for term in ["rmonth", "zebra_datepicker", "monthcoalreports"]
        )
    ]

    if matches:
        found = True
        print(f"\nScript block {index}:")
        for line_number, line in matches:
            print(f"Line {line_number}: {line[:500]}")

if not found:
    print("No relevant initialization found in inline scripts.")

print("\nNo form submission performed; no files downloaded.")

PAGE: https://npp.gov.in/monthlyCoalReports

MONTH FIELD:
{'name': 'rmonth', 'id': 'rmonth', 'type': 'text', 'class': ['gui-input', 'form-control'], 'placeholder': 'Select Month', 'readonly': '', 'aria-describedby': 'rmonth_help'}

INLINE JAVASCRIPT REFERENCES TO THE MONTH FIELD:

Script block 16:
Line 3: var $input = $('#rmonth');
Line 5: $('.Zebra_DatePicker:visible').find('td.dp_selected').css({
Line 48: $input.Zebra_DatePicker(monthPickerOptions);
Line 56: $('#rmonth').Zebra_DatePicker({
Line 62: $('#rmonth').val('');
Line 67: var rdate = $('#rmonth').val();

No form submission performed; no files downloaded.


In [ ]:
# [NB-00 | Cell 26] Read monthly archive JavaScript logic

import requests
from bs4 import BeautifulSoup

URL = "https://npp.gov.in/monthlyCoalReports"

response = requests.get(
    URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

scripts = soup.find_all("script")
matched = False

for index, script in enumerate(scripts, start=1):
    code = script.string or script.get_text()

    if not code or "#rmonth" not in code:
        continue

    matched = True
    lines = code.splitlines()

    print(f"SCRIPT BLOCK {index}")
    print("=" * 60)

    # Print the whole matching inline script if it is reasonably short
    if len(lines) <= 160:
        for line_number, line in enumerate(lines, start=1):
            print(f"{line_number:03d}: {line}")
    else:
        # Otherwise show context around each relevant statement
        for i, line in enumerate(lines):
            if any(term in line for term in [
                "#rmonth",
                "monthPickerOptions",
                "Zebra_DatePicker",
                "rdate",
                ".submit",
                "ajax",
                "csrf"
            ]):
                start = max(0, i - 5)
                end = min(len(lines), i + 9)
                print(f"\n--- Lines {start+1}–{end} ---")
                for j in range(start, end):
                    print(f"{j+1:03d}: {lines[j]}")

if not matched:
    print("No inline script containing '#rmonth' was found.")

print("\nInspection only. No form submitted and no files downloaded.")

SCRIPT BLOCK 16
001: 
002: 	$(document).ready(function() {
003: 		var $input = $('#rmonth');
004: 		function highlightSelectedPickerCell() {
005: 			$('.Zebra_DatePicker:visible').find('td.dp_selected').css({
006: 				backgroundColor: '#1f6feb',
007: 				color: '#ffffff',
008: 				border: '1px solid #174ea6'
009: 			});
010: 		}
011: 		var selectedMonth = '';
012: 		if (!selectedMonth) {
013: 			
014: 			selectedMonth = '';
015: 		}
016: 		if (selectedMonth) {
017: 			$input.val(selectedMonth);
018: 		}
019: 
020: 		var monthPickerOptions = {
021: 			format : 'm/Y',
022: 			view : 'months',
023: 			direction : -1,
024: 
025: 			show_icon : true,
026: 			inside : true,
027: 			readonly_element : true,
028: 			default_position : 'below',
029: 			offset : [0, 8],
030: 			container : $('body'),
031: 			show_clear_date : false,
032: 			onOpen : function() {
033: 				if (typeof window.nppPlaceReportDatepicker === 'function') {
034: 					window.nppPlaceReportDatepicker($input);
035: 				}
036

In [ ]:
# [NB-00 | Cell 27] Test official archive with the confirmed month format

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

URL = "https://npp.gov.in/monthlyCoalReports"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 StockRight research project"
})

# Obtain a fresh session and CSRF token
page = session.get(URL, timeout=30)
page.raise_for_status()

soup = BeautifulSoup(page.text, "html.parser")
csrf = soup.find("input", attrs={"name": "_csrf"})

if csrf is None or not csrf.get("value"):
    raise RuntimeError("Could not obtain a fresh CSRF token.")

test_month = "9/2022"

response = session.post(
    URL,
    data={
        "rmonth": test_month,
        "_csrf": csrf["value"]
    },
    headers={"Referer": URL},
    timeout=30
)
response.raise_for_status()

result = BeautifulSoup(response.text, "html.parser")
text_content = result.get_text(" ", strip=True)

print("TEST MONTH:", test_month)
print("HTTP status:", response.status_code)
print("Final URL:", response.url)

print("\nFORM FIELD AFTER SUBMISSION:")
field = result.find("input", attrs={"name": "rmonth"})
print(field.get("value") if field else "Field not found")

print("\nREPORT-LIKE LINKS:")
found_links = False

for link in result.find_all("a", href=True):
    href = urljoin(response.url, link["href"])
    label = link.get_text(" ", strip=True)

    if any(ext in href.lower() for ext in [".xls", ".xlsx", ".pdf"]):
        print(label or "(no link text)", "->", href)
        found_links = True

if not found_links:
    print("No direct XLS, XLSX, or PDF links found.")

print("\nPOSSIBLE STATUS MESSAGES:")
for phrase in [
    "Reporting Month Must Be Selected",
    "No Record",
    "No Data",
    "not available",
    "September",
    "Monthly Fuel Reports"
]:
    pos = text_content.lower().find(phrase.lower())
    if pos >= 0:
        print(text_content[max(0, pos-100):pos+250])

print("\nNo report files were saved or modified.")

TEST MONTH: 9/2022
HTTP status: 200
Final URL: https://npp.gov.in/monthlyCoalReports

FORM FIELD AFTER SUBMISSION:
None

REPORT-LIKE LINKS:
No direct XLS, XLSX, or PDF links found.

POSSIBLE STATUS MESSAGES:
n Reports Monthly Generation Reports Actual Monthly Generation Reports Tentative Daily Fuel Reports Monthly Fuel Reports Monthly Transmission Reports Installed Capacity Reports Home Monthly Fuel Reports Monthly
				Coal Reports Form Select Month Opens a calendar
							month picker. Submit

No report files were saved or modified.


In [ ]:
# [NB-00 | Cell 28] StockRight final raw-data inventory

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/StockRight")

folders = {
    "CEA daily coal reports": BASE / "raw" / "dailyCoal1",
    "India Data Portal": BASE / "raw" / "idp",
    "CEA DGR2 generation": BASE / "raw" / "dgr2",
    "CEA monthly coal reports": BASE / "raw" / "monthCoal1",
}

print("=" * 65)
print("STOCKRIGHT — RAW DATA INVENTORY")
print("=" * 65)

for label, folder in folders.items():
    files = sorted(
        f for f in folder.iterdir()
        if f.is_file()
    ) if folder.exists() else []

    print(f"\n{label}")
    print("Folder exists:", folder.exists())
    print("File count:", len(files))

    if files:
        print("File types:", {
            ext: sum(f.suffix.lower() == ext for f in files)
            for ext in sorted({f.suffix.lower() for f in files})
        })
        print("First file:", files[0].name)
        print("Last file:", files[-1].name)

print("\n" + "=" * 65)
print("DOWNLOAD LOG SUMMARY")
print("=" * 65)

log_files = [
    BASE / "logs" / "download_log.csv",
    BASE / "logs" / "dgr2_download_log.csv",
    BASE / "logs" / "monthCoal1_download_log.csv",
]

for log_path in log_files:
    print(f"\n{log_path.name}")

    if not log_path.exists():
        print("Log not found.")
        continue

    log = pd.read_csv(log_path)
    print("Entries:", len(log))

    if "status" in log.columns:
        print(log["status"].value_counts().to_string())

print("\n" + "=" * 65)
print("IDP CSV DETAILS")
print("=" * 65)

idp_path = BASE / "raw" / "idp" / "daily-coal-stocks.csv"

if idp_path.exists():
    print("File size (MB):", round(idp_path.stat().st_size / (1024**2), 2))

    idp = pd.read_csv(idp_path, usecols=["date"])
    dates = pd.to_datetime(idp["date"], errors="coerce").dropna()

    print("Records:", len(idp))
    print("Earliest date:", dates.min().date())
    print("Latest date:", dates.max().date())
    print("Unique report dates:", dates.dt.date.nunique())
else:
    print("IDP CSV not found.")

print("\nInventory check complete.")
print("No cleaning, EDA, forecasting, or modelling was performed.")

STOCKRIGHT — RAW DATA INVENTORY

CEA daily coal reports
Folder exists: True
File count: 835
File types: {'.xls': 835}
First file: dailyCoal1-2024-06-05.xls
Last file: dailyCoal1-2026-10-17.xls

India Data Portal
Folder exists: True
File count: 1
File types: {'.csv': 1}
First file: daily-coal-stocks.csv
Last file: daily-coal-stocks.csv

CEA DGR2 generation
Folder exists: True
File count: 1018
File types: {'.pdf': 3, '.xls': 1015}
First file: dgr2-2024-01-01.xls
Last file: dgr2-2026-10-14.xls

CEA monthly coal reports
Folder exists: True
File count: 41
File types: {'.xls': 41}
First file: monthCoal1-2023-Apr.xls
Last file: monthCoal1-2026-May.xls

DOWNLOAD LOG SUMMARY

download_log.csv
Entries: 1022
status
downloaded_xls     834
missing            187
already_present      1

dgr2_download_log.csv
Entries: 1022
status
downloaded_xls     1014
missing               4
downloaded_pdf        3
already_present       1

monthCoal1_download_log.csv
Entries: 105
status
missing            64
downlo

In [ ]:
# [NB-00 | Final Cell] Save the raw-data inventory

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/StockRight")
OUTPUT_DIR = BASE / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

datasets = [
    {
        "dataset": "CEA daily coal reports",
        "folder": "raw/dailyCoal1",
        "files": 835,
        "coverage": "2024-06-05 to 2026-10-17",
        "status": "Downloaded; 187 dates marked missing in log"
    },
    {
        "dataset": "India Data Portal daily coal stocks",
        "folder": "raw/idp",
        "files": 1,
        "coverage": "2018-11-27 to 2025-03-03",
        "status": "CSV downloaded; 362035 records"
    },
    {
        "dataset": "CEA DGR2 daily generation",
        "folder": "raw/dgr2",
        "files": 1018,
        "coverage": "2024-01-01 to 2026-10-14",
        "status": "Downloaded; 4 dates marked missing in log"
    },
    {
        "dataset": "CEA monthly coal reports",
        "folder": "raw/monthCoal1",
        "files": 41,
        "coverage": "2023-04 to 2026-05",
        "status": "Partial; 64 months not found in tested URLs"
    }
]

inventory = pd.DataFrame(datasets)
output_path = OUTPUT_DIR / "tab00_raw_inventory.csv"
inventory.to_csv(output_path, index=False)

print("Inventory saved successfully.")
print("Location:", output_path)
display(inventory)

Inventory saved successfully.
Location: /content/drive/MyDrive/StockRight/outputs/tables/tab00_raw_inventory.csv


,dataset,folder,files,coverage,status
0,CEA daily coal reports,raw/dailyCoal1,835,2024-06-05 to 2026-10-17,Downloaded; 187 dates marked missing in log
1,India Data Portal daily coal stocks,raw/idp,1,2018-11-27 to 2025-03-03,CSV downloaded; 362035 records
2,CEA DGR2 daily generation,raw/dgr2,1018,2024-01-01 to 2026-10-14,Downloaded; 4 dates marked missing in log
3,CEA monthly coal reports,raw/monthCoal1,41,2023-04 to 2026-05,Partial; 64 months not found in tested URLs
